In [1]:
import json
import os

os.environ["SPS_HOME"] = "/Users/z5114326/Documents/GitHub/python-fsps/src/fsps/libfsps"

import fsps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fsps.filters import FILTERS

In [2]:
dat_dir = "../data/"
ref_band = "v"

# Restart kernel and run first section for both band_types

In [3]:
band_type = "SDSS"  # JC is vega, SDSS is AB
default_age = 12  # gyr

In [4]:
bands_path = os.path.join(dat_dir, "supplementary/bands.json")
with open(bands_path, "r") as f:
    bands_dict = json.load(f)

mw_path = os.path.join(dat_dir, "mw/gcs/mw_gcs.csv")
mw_gcs = pd.read_csv(mw_path)

bands = list(bands_dict[band_type].keys())
bands_fsps = [bands_dict[band_type][band]["fsps"] for band in bands]

assert (band_type == "JC") | (band_type == "SDSS"), "band_type must be JC or SDSS"

if band_type == "JC":
    compute_vega_mags = True
elif band_type == "SDSS":
    compute_vega_mags = False

sp = fsps.StellarPopulation(imf_type=2, zcontinuous=1, compute_vega_mags=compute_vega_mags)
sp.params["add_dust_emission"] = False
sp.params["dust1"] = 0
sp.params["dust2"] = 0
sp.params["fcstar"] = 0.0
sp.params["add_neb_continuum"] = False
sp.params["add_neb_emission"] = False
sp.params["add_agb_dust_model"] = False
sp.params["agb_dust"] = 0.0
sp.params["tpagb_norm_type"] = 0

ref_path = os.path.join(dat_dir, "mw/colors/", "mag_reference.csv")
if not os.path.exists(ref_path):
    # create new DataFrame (define columns if needed)
    df = pd.DataFrame()
else:
    # load existing CSV
    df = pd.read_csv(ref_path)

cluster_name = np.array(mw_gcs["cluster_name"])
age_msk = np.array(mw_gcs["age_assume"].astype(bool))
ages = np.array(mw_gcs["age_gyr"])
ages[age_msk] = default_age
fehs = np.array(mw_gcs["feh"])

df["cluster_name"] = cluster_name

df_dict = {}
for band in bands_fsps:
    df_dict[band] = []
    for age, feh in zip(ages, fehs):
        if np.isnan(feh):
            df_dict[band].append(np.nan)
            continue
        sp.params["logzsol"] = feh
        mag = sp.get_mags(tage=age, bands=[band])[0]
        df_dict[band].append(mag)

for band in df_dict.keys():
    df[band] = df_dict[band]

df.to_csv(ref_path, index=False)

# Create color reference csv

In [ ]:
bands_path = os.path.join(dat_dir, "supplementary/bands.json")
with open(bands_path, "r") as f:
    bands_dict = json.load(f)

ref_path = os.path.join(dat_dir, "mw/colors/", "mag_reference.csv")
df_mag = pd.read_csv(ref_path)

rev_dict = {}
for band_type in bands_dict.keys():
    for band in bands_dict[band_type].keys():
        band_key = band_type + "_" + band
        rev_dict[bands_dict[band_type][band]["fsps"]] = band_key

col_path = os.path.join(dat_dir, "mw/colors/", "colors_reference.csv")
if not os.path.exists(col_path):
    # create new DataFrame (define columns if needed)
    df_col = pd.DataFrame()
else:
    # load existing CSV
    df_col = pd.read_csv(col_path)

df_col["cluster_name"] = df_mag["cluster_name"]

for column in df_mag.columns:
    if (column == "cluster_name") or (column == ref_band):
        continue

    color = df_mag[ref_band] - df_mag[column]
    new_col_name = rev_dict[ref_band] + "-" + rev_dict[column]
    df_col[new_col_name] = color

df_col.to_csv(col_path, index=False)